In [579]:
import numpy as np
import pandas as pd

df_2021 = pd.read_csv(r"C:\Users\david\Downloads\temp_offerings_2021_anon.tsv", sep="\t")
df_2022 = pd.read_csv(r"C:\Users\david\Downloads\temp_offerings_2022_anon.tsv", sep="\t")
df_2023 = pd.read_csv(r"C:\Users\david\Downloads\temp_offerings_2023_anon.tsv", sep="\t")
df_2024 = pd.read_csv(r"C:\Users\david\Downloads\temp_offerings_2024_anon.tsv", sep="\t")
price = pd.read_csv(r"C:\Users\david\Downloads\temp_prices_2021_2024_anon.tsv", sep="\t")

In [580]:
news = pd.concat([df_2021, df_2022, df_2023, df_2024], ignore_index=True)

In [581]:
news["timestamp"] = pd.to_datetime(news["timestamp"])

news["date"] = news["timestamp"].dt.date
news["time"] = news["timestamp"].dt.time

In [582]:
news.head()

,timestamp,symbol,headline,body,date,time
0,2021-01-04 08:06:42,FWOM,Happiness Biotech Announces Registered Direct ...,"NANPING, China, Jan. 4, 2021 /PRNewswire/ -- H...",2021-01-04,08:06:42
1,2021-01-04 09:27:19,HTYA,AgEagle Aerial Systems Prices $6.375M Register...,"WICHITA, Kan., Jan. 04, 2021 (GLOBE NEWSWIRE) ...",2021-01-04,09:27:19
2,2021-01-04 16:02:09,ZUOY,Fate Therapeutics Announces Proposed Offering ...,Fate Therapeutics Announces Proposed Public Of...,2021-01-04,16:02:09
3,2021-01-04 18:43:14,HUHZ,AdaptHealth Announced 7M Share Proposed Public...,AdaptHealth Corp. (NASDAQ: HUHZ ) announced to...,2021-01-04,18:43:14
4,2021-01-05 12:10:36,RCJI,ProPhase Labs Announces $5.5M Offering Of Stoc...,ProPhase Labs Announces $5.5 Million Registere...,2021-01-05,12:10:36


In [583]:
from datetime import time

market_open = time(9, 30)

news = news[news["time"] < market_open]

In [584]:
news["text"] = news["headline"].fillna("") + " " + news["body"].fillna("")

In [585]:
grouped_news = (
    news.groupby(["symbol", "date"])["text"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

In [586]:
grouped_news.head()

,symbol,date,text
0,ABJ,2024-02-14,CORRECTION: Ohmyhome Prices Upsized $4.8M Publ...
1,ABYR,2021-02-09,Ekso Bionics Raises $40M From Upsized Equity O...
2,ACEV,2021-02-02,Sundial Growers Reports $74.5M Offering Of 65M...
3,ACI,2021-06-22,"GameStop Raises Over $1B In Sale Of 5M Shares,..."
4,ACI,2024-06-07,GameStop Plans To Sell Up To 75M Shares Of Com...


In [587]:
price["date"] = pd.to_datetime(price["date"]).dt.date

In [588]:
price["return"] = (price["close"] - price["open"]) / price["open"]

In [589]:
price.head()

,date,symbol,open,high,low,close,volume,return
0,2021-11-01,KHEQ,27.04,28.0000,26.60,27.53,89941,0.018121
1,2021-11-02,KHEQ,27.50,28.7560,26.62,28.04,77635,0.019636
2,2021-11-03,KHEQ,28.00,28.6272,28.00,28.06,77256,0.002143
3,2021-11-04,KHEQ,28.46,29.1000,27.76,29.10,59481,0.022488
4,2021-11-05,KHEQ,28.45,28.9699,27.23,27.58,37246,-0.030580


In [590]:
data = grouped_news.merge(price, on=["symbol", "date"])

In [591]:
data = data.merge(
    news[["symbol", "date", "headline", "body"]],
    on=["symbol", "date"],
    how="left"
)

In [592]:
data.head()

,symbol,date,text,open,high,low,close,volume,return,headline,body
0,ABJ,2024-02-14,CORRECTION: Ohmyhome Prices Upsized $4.8M Publ...,1.5800,1.7400,1.20,1.3800,5734804,-0.126582,CORRECTION: Ohmyhome Prices Upsized $4.8M Publ...,"Ohmyhome Ltd. (NASDAQ: ABJ , ""Ohmyhome""))))), ..."
1,ACI,2024-06-07,GameStop Plans To Sell Up To 75M Shares Of Com...,61.9000,62.7900,26.12,27.1600,263260841,-0.561228,GameStop Plans To Sell Up To 75M Shares Of Com...,"On June 7, 2024, GameStop Corp., a Delaware co..."
2,ACK,2024-09-23,Pebblebrook Hotel Trust To Raise $350M Via Pro...,14.2800,14.4900,13.63,13.9600,1899285,-0.022409,Pebblebrook Hotel Trust To Raise $350M Via Pro...,RepaymenrtPebblebrook Hotel Trust (NYSE: ACK )...
3,ADU,2024-01-30,Palatin Announces $10M Registered Direct Offer...,5.4000,5.5300,4.00,4.8800,900793,-0.096296,Palatin Announces $10M Registered Direct Offer...,"Palatin Technologies, Inc. (NYSE: ADU ) (&quot..."
4,ADXY,2022-09-26,SiNtx Technologies Announces Commencement Of R...,0.4009,0.4046,0.33,0.3377,198211,-0.157645,SiNtx Technologies Announces Commencement Of R...,"SALT SVAX CITY, UT, Sept. 26, 2022 (GLOBE NEWS..."


In [593]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X = vectorizer.fit_transform(data["text"])
y = data["return"]

In [595]:
data["date"] = pd.to_datetime(data["date"])

train = data[data["date"] < "2023-01-01"]
test  = data[data["date"] >= "2023-01-01"]

X_train = vectorizer.fit_transform(train["text"])
X_test  = vectorizer.transform(test["text"])

y_train = train["return"]
y_test  = test["return"]

In [596]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

preds = model.predict(X_test)

In [597]:
from sklearn.metrics import mean_squared_error
mean_squared_error(y_test, preds)

0.03366524101773109

In [598]:
direction_acc = ((preds > 0) == (y_test > 0)).mean()
print(f"Direction Accuracy: {direction_acc:.2%}")

Direction Accuracy: 63.26%


In [599]:
np.corrcoef(preds, y_test)[0,1]

np.float64(0.3779594271342188)

In [600]:
strategy_returns = np.sign(preds) * y_test

sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
print(f"Strategy Sharpe Ratio: {sharpe:.2f}")

Strategy Sharpe Ratio: 5.72


In [601]:
from sklearn.linear_model import Ridge
import numpy as np

alphas = [0, 0.01, 0.1, 1, 10, 100]

results = []

for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    corr = np.corrcoef(preds, y_test)[0,1]
    
    direction_acc = ((preds > 0) == (y_test > 0)).mean()

    strategy_returns = np.sign(preds) * y_test
    sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
    
    results.append((a, mse, corr, direction_acc, sharpe))

results_df = pd.DataFrame(results, columns=["alpha", "mse", "corr", "direction_acc", "sharpe"])
print(results_df)

    alpha       mse      corr  direction_acc    sharpe
0    0.00  0.040569  0.325703       0.630013  5.040240
1    0.01  0.039974  0.329094       0.628719  4.971553
2    0.10  0.037223  0.347163       0.623545  5.132623
3    1.00  0.033665  0.377959       0.632600  5.719355
4   10.00  0.035897  0.386199       0.640362  4.793955
5  100.00  0.038645  0.367813       0.640362  4.793955


In [602]:
data = data.sort_values(["symbol", "date"])

data["prev_return"] = data.groupby("symbol")["return"].shift(1)

In [603]:
data = data.dropna(subset=["prev_return"])

In [604]:
data["date"] = pd.to_datetime(data["date"])

train = data[data["date"] < "2023-01-01"]
test  = data[data["date"] >= "2023-01-01"]

In [605]:
data.head()

,symbol,date,text,open,high,low,close,volume,return,headline,body,prev_return
5,ADXY,2024-03-22,SINTX Technologies Announces Proposed Public O...,0.136,0.1417,0.0824,0.0926,7033183,-0.319118,SINTX Technologies Announces Proposed Public O...,"SINTX Technologies, Inc. (NASDAQ: ADXY ) (""SIN...",-0.157645
6,ADXY,2024-04-03,SINTX Technologies Announces Pricing Of $1.5M ...,0.031,0.0397,0.0200,0.0261,83529750,-0.158065,SINTX Technologies Announces Pricing Of $1.5M ...,"SINTX Technologies, Inc. (NASDAQ: ADXY ) (""SIN...",-0.319118
12,AGAU,2023-05-19,Ault Alliance Announces Termination of Exchang...,16.700,17.5000,13.8000,14.7000,96143,-0.119760,Ault Alliance Announces Termination of Exchang...,"Ault Alliance, Inc announced today that it has...",0.223464
16,AICR,2024-08-22,Tevogen Bio Secures $6M Series C Preferred Sto...,0.669,0.6690,0.5795,0.5901,165227,-0.117937,Tevogen Bio Secures $6M Series C Preferred Sto...,"Tevogen Bio Holdings Inc. (""Tevogen"" or ""Tevog...",0.934091
22,AL,2024-07-29,Permian Resources Prices Public Offering Of 26...,15.270,15.5000,14.7000,15.3000,32055290,0.001965,Permian Resources Prices Public Offering Of 26...,"Permian Resources Corporation (""Permian Resour...",-0.006463


In [606]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train_text = vectorizer.fit_transform(train["text"])
X_test_text  = vectorizer.transform(test["text"])

In [607]:
from scipy.sparse import hstack
import numpy as np

X_train = hstack([X_train_text, np.array(train["prev_return"]).reshape(-1,1)])
X_test  = hstack([X_test_text, np.array(test["prev_return"]).reshape(-1,1)])

In [608]:
y_train = (train["return"] > 0).astype(int)
y_test  = (test["return"] > 0).astype(int)

In [609]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:,1]
preds = (probs > 0.5).astype(int)

In [610]:
accuracy = (preds == y_test).mean()
print(f"Direction Accuracy: {accuracy:.2%}")

Direction Accuracy: 71.72%


In [611]:
positions = np.where(preds == 1, 1, -1)
strategy_returns = positions * test["return"].values

In [612]:
sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
print(f"Sharpe Ratio: {sharpe:.2f}")

Sharpe Ratio: 7.21
